# Gatekeeper, giai đoạn 0 (§37), phần A: Kaggle có chấm được task SWE-smith với cùng điểm không?

**Trước khi chạy** (khung Settings bên phải): **Internet: On**, **Accelerator: None**.

Chạy lần lượt ô 1 đến ô 4. **Mỗi ô tự đủ**: nếu kernel bị khởi động lại, chỉ cần chạy lại ô 1 rồi tiếp tục ở ô đang dở.
Ô 3 và ô 4 chạy việc nặng trong một tiến trình riêng và in tiến độ ngay khi có.
Không ô nào in khóa hay thông tin tài khoản. Kết quả ở `/kaggle/working/`: `k0a.json`, `k0b_kaggle.jsonl`.

In [ ]:
# Ô 1 — cài đặt vào đúng Python của notebook (khoảng 2–3 phút; chạy lại an toàn)
%env UDOCKER_DIR=/kaggle/working/udocker
%pip -q install "swesmith==0.0.9" "swebench==4.1.0" udocker unidiff
import subprocess  # noqa: E402
import sys  # noqa: E402

UD = "import sys; from udocker.maincmd import main; sys.argv[0] = 'udocker'; sys.exit(main())"
r = subprocess.run(
    [sys.executable, "-c", UD, "--allow-root", "install"], capture_output=True, text=True
)
print("udocker install:", r.returncode, (r.stdout + r.stderr).strip()[-200:])
!docker info > /dev/null 2>&1 && echo "Docker daemon: CO" || echo "Docker daemon: KHONG"
!df -h /kaggle/working | tail -1
print("python:", sys.executable)

In [ ]:
# ruff: noqa: F811 (every cell redefines the same helpers on purpose: a cell must run alone)
import os
import subprocess
import sys
import time

W = "/kaggle/working"
os.environ["UDOCKER_DIR"] = f"{W}/udocker"
CODE = next(
    (
        f"{W}/{d}"
        for d in ("gatekeeper/kaggle/scripts", "gatekeeper/scripts")
        if os.path.exists(f"{W}/{d}/udocker_shim.py")
    ),
    None,
)


def _mem_avail_gb():
    for line in open("/proc/meminfo"):
        if line.startswith("MemAvailable"):
            return round(int(line.split()[1]) / 2**20, 1)


def run_bg(name, cmd):
    """Run a step fully detached from this kernel, log to f"{W}/{name}.log", and follow the log
    with a heartbeat every 20 s. Re-running the cell after a kernel restart does not start the
    step again while it runs or once it finished: it only shows the log."""
    log, pidf, donef = f"{W}/{name}.log", f"{W}/{name}.pid", f"{W}/{name}.done"
    alive = False
    if os.path.exists(pidf):
        try:
            os.kill(int(open(pidf).read()), 0)
            alive = True
        except (OSError, ValueError):
            alive = False
    if not alive and not os.path.exists(donef):
        wrapped = f'{" ".join(cmd)}; echo "exit code: $?" ; touch {donef}'
        p = subprocess.Popen(
            ["/bin/bash", "-c", wrapped],
            stdout=open(log, "a"),
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        open(pidf, "w").write(str(p.pid))
        print(f"started {name} (pid {p.pid}); log: {log}", flush=True)
    else:
        print(
            f"{name}: {'still running' if alive else 'already finished'}; showing log", flush=True
        )
    t0, pos = time.time(), 0
    while True:
        with open(log) as fh:
            fh.seek(pos)
            new = fh.read()
            pos = fh.tell()
        if new:
            print(new, end="", flush=True)
        if os.path.exists(donef):
            break
        st = os.statvfs(W)
        used = round((st.f_blocks - st.f_bfree) * st.f_frsize / 2**30, 1)
        secs = int(time.time() - t0)
        print(f"  [{secs} s] working used {used} GB, RAM free {_mem_avail_gb()} GB", flush=True)
        time.sleep(20)


# Ô 2 — mã và dữ liệu (repo công khai; không cần đăng nhập)
REPO = "dinhthithuthuyduphong-bot/gatekeeper"
BRANCH = "claude/gatekeeper-13-research-nxytoh"  # Kaggle work lives here until it runs
os.chdir(W)
if not os.path.isdir("gatekeeper"):
    url = f"https://github.com/{REPO}"
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url], check=True)
else:
    subprocess.run(["git", "-C", "gatekeeper", "pull", "-q"], check=False)
RAW = f"https://raw.githubusercontent.com/{REPO}"
files = {
    "swesmith_pallets__jinja.jsonl": "data-snapshots/data/instances/swesmith_pallets__jinja.jsonl",
    "f1_probe.json": "trial/results/f1_probe.json",
    "f1_skill_probe.json": "trial/results/f1_skill_probe.json",
}
os.makedirs("k0data", exist_ok=True)
for name, path in files.items():
    dest = f"k0data/{name}"
    if not os.path.exists(dest):
        subprocess.run(["curl", "-sSfL", "-o", dest, f"{RAW}/{path}"], check=True)
    print(name, os.path.getsize(dest), "byte")
head = subprocess.run(
    ["git", "-C", "gatekeeper", "log", "--oneline", "-1"], capture_output=True, text=True
)
print(head.stdout)
CODE = f"{W}/gatekeeper/kaggle/scripts"
print("code:", CODE, os.path.exists(f"{CODE}/k0a_probe.py"))

In [ ]:
import os
import subprocess
import sys
import time

W = "/kaggle/working"
os.environ["UDOCKER_DIR"] = f"{W}/udocker"
CODE = next(
    (
        f"{W}/{d}"
        for d in ("gatekeeper/kaggle/scripts", "gatekeeper/scripts")
        if os.path.exists(f"{W}/{d}/udocker_shim.py")
    ),
    None,
)


def _mem_avail_gb():
    for line in open("/proc/meminfo"):
        if line.startswith("MemAvailable"):
            return round(int(line.split()[1]) / 2**20, 1)


def run_bg(name, cmd):
    """Run a step fully detached from this kernel, log to f"{W}/{name}.log", and follow the log
    with a heartbeat every 20 s. Re-running the cell after a kernel restart does not start the
    step again while it runs or once it finished: it only shows the log."""
    log, pidf, donef = f"{W}/{name}.log", f"{W}/{name}.pid", f"{W}/{name}.done"
    alive = False
    if os.path.exists(pidf):
        try:
            os.kill(int(open(pidf).read()), 0)
            alive = True
        except (OSError, ValueError):
            alive = False
    if not alive and not os.path.exists(donef):
        wrapped = f'{" ".join(cmd)}; echo "exit code: $?" ; touch {donef}'
        p = subprocess.Popen(
            ["/bin/bash", "-c", wrapped],
            stdout=open(log, "a"),
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        open(pidf, "w").write(str(p.pid))
        print(f"started {name} (pid {p.pid}); log: {log}", flush=True)
    else:
        print(
            f"{name}: {'still running' if alive else 'already finished'}; showing log", flush=True
        )
    t0, pos = time.time(), 0
    while True:
        with open(log) as fh:
            fh.seek(pos)
            new = fh.read()
            pos = fh.tell()
        if new:
            print(new, end="", flush=True)
        if os.path.exists(donef):
            break
        st = os.statvfs(W)
        used = round((st.f_blocks - st.f_bfree) * st.f_frsize / 2**30, 1)
        secs = int(time.time() - t0)
        print(f"  [{secs} s] working used {used} GB, RAM free {_mem_avail_gb()} GB", flush=True)
        time.sleep(20)


# Ô 3 — K0a: image của task có chạy được không; thử các cách chạy của udocker (5–15 phút)
# Chạy tách khỏi notebook. Nếu kernel bị ngắt: chạy lại ô 1 rồi ô 3, ô 3 chỉ hiện tiếp log.
IMAGE = "swebench/swesmith.x86_64.pallets_1776_jinja.ada0a9a6"
run_bg("k0a", [sys.executable, f"{CODE}/k0a_probe.py", "--image", IMAGE, "--out-dir", W])

In [ ]:
import os
import subprocess
import sys
import time

W = "/kaggle/working"
os.environ["UDOCKER_DIR"] = f"{W}/udocker"
CODE = next(
    (
        f"{W}/{d}"
        for d in ("gatekeeper/kaggle/scripts", "gatekeeper/scripts")
        if os.path.exists(f"{W}/{d}/udocker_shim.py")
    ),
    None,
)


def _mem_avail_gb():
    for line in open("/proc/meminfo"):
        if line.startswith("MemAvailable"):
            return round(int(line.split()[1]) / 2**20, 1)


def run_bg(name, cmd):
    """Run a step fully detached from this kernel, log to f"{W}/{name}.log", and follow the log
    with a heartbeat every 20 s. Re-running the cell after a kernel restart does not start the
    step again while it runs or once it finished: it only shows the log."""
    log, pidf, donef = f"{W}/{name}.log", f"{W}/{name}.pid", f"{W}/{name}.done"
    alive = False
    if os.path.exists(pidf):
        try:
            os.kill(int(open(pidf).read()), 0)
            alive = True
        except (OSError, ValueError):
            alive = False
    if not alive and not os.path.exists(donef):
        wrapped = f'{" ".join(cmd)}; echo "exit code: $?" ; touch {donef}'
        p = subprocess.Popen(
            ["/bin/bash", "-c", wrapped],
            stdout=open(log, "a"),
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        open(pidf, "w").write(str(p.pid))
        print(f"started {name} (pid {p.pid}); log: {log}", flush=True)
    else:
        print(
            f"{name}: {'still running' if alive else 'already finished'}; showing log", flush=True
        )
    t0, pos = time.time(), 0
    while True:
        with open(log) as fh:
            fh.seek(pos)
            new = fh.read()
            pos = fh.tell()
        if new:
            print(new, end="", flush=True)
        if os.path.exists(donef):
            break
        st = os.statvfs(W)
        used = round((st.f_blocks - st.f_bfree) * st.f_frsize / 2**30, 1)
        secs = int(time.time() - t0)
        print(f"  [{secs} s] working used {used} GB, RAM free {_mem_avail_gb()} GB", flush=True)
        time.sleep(20)


# Ô 4 — K0b: chấm lại các bản vá đã lưu, so với điểm đã ghi dưới Docker
# (30–60 phút; chạy tách khỏi notebook; các lượt đã chấm được bỏ qua khi chạy lại)
mode = open(f"{W}/k0a_mode.txt").read().strip() if os.path.exists(f"{W}/k0a_mode.txt") else ""
if not mode:
    raise SystemExit("Ô 3 chưa tìm được cách chạy nào (k0a_mode.txt rỗng): gửi k0a.log, k0a.json")
os.environ["UDOCKER_EXECMODE"] = mode
print("udocker execution mode:", mode)
os.makedirs(f"{W}/k0run", exist_ok=True)
os.chdir(f"{W}/k0run")  # SWE-smith writes its logs under ./logs/run_evaluation
D = f"{W}/k0data"
args = [sys.executable, f"{CODE}/k0_stage0.py", "--rows", f"{D}/swesmith_pallets__jinja.jsonl"]
args += ["--results", f"{D}/f1_probe.json", f"{D}/f1_skill_probe.json"]
args += ["--controls", "20", "--out", f"{W}/k0b_kaggle.jsonl"]
run_bg("k0b", args + ["&&", "rm", "-rf", f"{W}/k0run/logs"])

## Sau khi chạy xong

Gửi lại `/kaggle/working/k0a.log`, `k0a.json`, `k0b.log` và `k0b_kaggle.jsonl` (khung Output, nút Download), cùng phần in ra của ô 3 và vài dòng cuối của ô 4.

Phần B (K0c, K0d: gọi model bằng hạn mức AI) chạy ở notebook tạo từ https://www.kaggle.com/benchmarks/tasks/new, bằng ô `kaggle/notebooks/k0cd_benchmarks_cell.py`.